# LeetCode 1906 — Minimum Absolute Difference Queries bằng Python

Link đề: https://leetcode.com/problems/minimum-absolute-difference-queries/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minDifference(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        ...
```

Lộ trình:
1. **Brute force cặp** trong từng đoạn — `O(len²)` mỗi truy vấn.
2. **Sắp xếp từng đoạn**, so các phần tử kề khác nhau — `O(len log len)` mỗi truy vấn.
3. **Prefix count theo giá trị** (`nums[i] ≤ 100`): biết giá trị nào có mặt trong đoạn bằng `O(1)` ⇒ `O(100)` mỗi truy vấn.
4. **Danh sách vị trí + `bisect`** cho từng giá trị — cùng ý tưởng, bộ nhớ `O(n)` (**nộp bài**).
5. **Segment tree bitmask**: mỗi nút lưu tập giá trị có mặt dưới dạng số nguyên 100 bit; OR theo đoạn.

## 1. Mục tiêu học

1. Nhận ra **miền giá trị nhỏ** (`1..100`) là chìa khóa: thay vì sắp xếp đoạn, chỉ cần biết **tập giá trị có mặt**.
2. Dựng **prefix count theo từng giá trị** và dùng nó để kiểm tra "giá trị `v` có trong `nums[l..r]`" trong `O(1)`.
3. Hiểu vì sao đáp án là **khoảng cách nhỏ nhất giữa hai giá trị có mặt liên tiếp** (theo thứ tự tăng), và trả `-1` khi chỉ có một giá trị.
4. Biết các phương án thay thế: `bisect` trên danh sách vị trí, segment tree lưu bitmask (hỗ trợ cả cập nhật).
5. Tránh lỗi off-by-one của prefix (`pre[r + 1] - pre[l]`), lỗi khởi tạo `prev`, lỗi tính cả hiệu `0`.

## 2. Đề bài & ví dụ

**Hiệu tuyệt đối nhỏ nhất** của mảng `a` là giá trị nhỏ nhất của `|a[i] - a[j]|` với `i < j` và `a[i] ≠ a[j]`. Nếu mọi phần tử bằng nhau thì kết quả là `-1`.

Cho `nums` và `queries[i] = [l, r]`. Với mỗi truy vấn, tính hiệu tuyệt đối nhỏ nhất của đoạn con `nums[l..r]` (chỉ số từ 0, gồm cả hai đầu).

**Ví dụ 1:** `nums = [1,3,4,8]`, `queries = [[0,1],[1,2],[2,3],[0,3]]` → `[2,1,4,1]`.

**Ví dụ 2:** `nums = [4,5,2,2,7,10]`, `queries = [[2,3],[0,2],[0,5],[3,5]]` → `[-1,1,1,3]` (đoạn `[2,2]` toàn giá trị bằng nhau ⇒ `-1`).

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 100`, `1 ≤ q ≤ 2·10⁴`, `0 ≤ l < r < n`.

In [1]:
import random
import time
from bisect import bisect_left
from itertools import accumulate
from typing import List

examples = [
    (([1, 3, 4, 8], [[0, 1], [1, 2], [2, 3], [0, 3]]), [2, 1, 4, 1]),
    (([4, 5, 2, 2, 7, 10], [[2, 3], [0, 2], [0, 5], [3, 5]]), [-1, 1, 1, 3]),
]
for (nums, qs), want in examples:
    for (l, r), w in zip(qs, want):
        print(f"nums[{l}..{r}] = {nums[l:r + 1]} -> {w}")
    print()

nums[0..1] = [1, 3] -> 2
nums[1..2] = [3, 4] -> 1
nums[2..3] = [4, 8] -> 4
nums[0..3] = [1, 3, 4, 8] -> 1

nums[2..3] = [2, 2] -> -1
nums[0..2] = [4, 5, 2] -> 1
nums[0..5] = [4, 5, 2, 2, 7, 10] -> 1
nums[3..5] = [2, 7, 10] -> 3



## 3. Kiến thức nền

### 3.1 Hiệu nhỏ nhất của một mảng = hiệu nhỏ nhất giữa hai giá trị **phân biệt** kề nhau sau khi sắp xếp

Nếu `x < y < z` thì `z - x > y - x` — nên cặp tối ưu luôn là hai giá trị liền nhau trong **tập giá trị đã sắp xếp**.

In [2]:
def min_abs_diff(arr):
    vals = sorted(set(arr))
    if len(vals) < 2:
        return -1
    return min(b - a for a, b in zip(vals, vals[1:]))

for arr in ([5, 2, 3, 7, 2], [2, 2], [1, 3, 4, 8], [10, 1, 10]):
    print(arr, "->", min_abs_diff(arr))

[5, 2, 3, 7, 2] -> 1
[2, 2] -> -1
[1, 3, 4, 8] -> 1
[10, 1, 10] -> 9


### 3.2 Prefix count: đếm số lần `v` xuất hiện trong đoạn

`pre[v][i]` = số chỉ số `k < i` có `nums[k] = v`. Khi đó số lần `v` trong `nums[l..r]` = `pre[v][r + 1] - pre[v][l]`.

In [3]:
nums = [4, 5, 2, 2, 7, 10]
pre2 = [0] + list(accumulate(1 if x == 2 else 0 for x in nums))
print("pre[2] =", pre2)
for l, r in [(2, 3), (0, 1), (3, 5)]:
    print(f"số lần 2 trong nums[{l}..{r}] = pre[2][{r + 1}] - pre[2][{l}] = {pre2[r + 1] - pre2[l]}")

pre[2] = [0, 0, 0, 1, 2, 2, 2]
số lần 2 trong nums[2..3] = pre[2][4] - pre[2][2] = 2
số lần 2 trong nums[0..1] = pre[2][2] - pre[2][0] = 0
số lần 2 trong nums[3..5] = pre[2][6] - pre[2][3] = 1


### 3.3 `bisect` trên danh sách vị trí

`pos[v]` = danh sách tăng dần các chỉ số có giá trị `v`. Giá trị `v` có mặt trong `[l, r]` ⇔ phần tử đầu tiên `≥ l` của `pos[v]` tồn tại và `≤ r`.

In [4]:
pos = {}
for i, x in enumerate(nums):
    pos.setdefault(x, []).append(i)
print(pos)

def present(v, l, r):
    p = pos.get(v, [])
    k = bisect_left(p, l)
    return k < len(p) and p[k] <= r

print([v for v in range(1, 11) if present(v, 3, 5)])

{4: [0], 5: [1], 2: [2, 3], 7: [4], 10: [5]}
[2, 7, 10]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 2, bảng "giá trị nào có mặt trong đoạn" (chỉ liệt kê các giá trị xuất hiện trong `nums`):

| đoạn | 2 | 4 | 5 | 7 | 10 | giá trị có mặt (tăng dần) | hiệu kề | đáp án |
|---|---|---|---|---|---|---|---|---|
| `[2,3]` | ✓ | | | | | 2 | — | `-1` |
| `[0,2]` | ✓ | ✓ | ✓ | | | 2, 4, 5 | 2, 1 | `1` |
| `[0,5]` | ✓ | ✓ | ✓ | ✓ | ✓ | 2, 4, 5, 7, 10 | 2, 1, 2, 3 | `1` |
| `[3,5]` | ✓ | | | ✓ | ✓ | 2, 7, 10 | 5, 3 | `3` |

Chỉ cần **tập giá trị** — không cần thứ tự hay số lần xuất hiện. Vì giá trị chỉ từ `1` tới `100`, duyệt `v = 1..100` mỗi truy vấn là đủ rẻ: `2·10⁴ · 100 = 2·10⁶` bước.

In [5]:
nums, qs = examples[1][0]
for l, r in qs:
    have = sorted(set(nums[l:r + 1]))
    gaps = [b - a for a, b in zip(have, have[1:])]
    print(f"[{l},{r}] có mặt {have}, hiệu kề {gaps} -> {min(gaps) if gaps else -1}")

[2,3] có mặt [2], hiệu kề [] -> -1
[0,2] có mặt [2, 4, 5], hiệu kề [2, 1] -> 1
[0,5] có mặt [2, 4, 5, 7, 10], hiệu kề [2, 1, 2, 3] -> 1
[3,5] có mặt [2, 7, 10], hiệu kề [5, 3] -> 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

**Ý tưởng:** với mỗi truy vấn, thử mọi cặp `i < j` trong `[l, r]` có `nums[i] ≠ nums[j]`.

In [6]:
def min_diff_brute(nums, queries):
    out = []
    for l, r in queries:
        best = -1
        for i in range(l, r + 1):
            for j in range(i + 1, r + 1):
                if nums[i] != nums[j]:
                    d = abs(nums[i] - nums[j])
                    if best == -1 or d < best:
                        best = d
        out.append(best)
    return out

for (nums, qs), want in examples:
    print(min_diff_brute(nums, qs), "kỳ vọng", want)

[2, 1, 4, 1] kỳ vọng [2, 1, 4, 1]
[-1, 1, 1, 3] kỳ vọng [-1, 1, 1, 3]


**Độ phức tạp:** `O(Σ len²)` — tệ nhất `2·10⁴ · (10⁵)² / 2` — không khả thi. Space `O(1)` ngoài output.

**Nút thắt:** so mọi cặp, trong khi chỉ cần các giá trị kề nhau sau khi sắp xếp (mục 3.1).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mục 3.1):** đáp án chỉ phụ thuộc **tập giá trị có mặt** `S(l, r)`: nếu `|S| < 2` thì `-1`, ngược lại là hiệu nhỏ nhất của hai phần tử kề nhau trong `S` sắp xếp.

**Insight 2:** `S(l, r) ⊆ {1, …, 100}` ⇒ duyệt `v` tăng dần từ `1` tới `100`, kiểm tra `v ∈ S(l, r)` bằng prefix count trong `O(1)`; giữ `prev` = giá trị có mặt gần nhất trước đó, cập nhật `best = min(best, v - prev)`.

Không cần sắp xếp đoạn: duyệt `v` tăng dần **chính là** sắp xếp đếm (counting sort) trên miền nhỏ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp từng đoạn

In [7]:
def min_diff_sort(nums, queries):
    return [min_abs_diff(nums[l:r + 1]) for l, r in queries]

for (nums, qs), want in examples:
    print(min_diff_sort(nums, qs), "kỳ vọng", want)

[2, 1, 4, 1] kỳ vọng [2, 1, 4, 1]
[-1, 1, 1, 3] kỳ vọng [-1, 1, 1, 3]


**Độ phức tạp:** `O(Σ len · log len)` — tệ nhất `2·10⁴ · 10⁵ · 17 ≈ 3·10¹⁰`. Quá chậm khi các đoạn dài. Space `O(len)`.

Cải tiến nhỏ: thay `sorted(set(...))` bằng mảng đánh dấu 101 phần tử ⇒ `O(len + 100)` mỗi truy vấn, vẫn `O(q · n)` tệ nhất.

### 7.2 Cách 3 — Prefix count theo giá trị

In [8]:
def min_diff_prefix(nums, queries):
    vals = sorted(set(nums))                                    # chỉ những giá trị thật sự xuất hiện
    pre = {v: [0] + list(accumulate(1 if x == v else 0 for x in nums)) for v in vals}
    out = []
    for l, r in queries:
        best, prev = 101, -1
        for v in vals:
            p = pre[v]
            if p[r + 1] - p[l] > 0:
                if prev != -1 and v - prev < best:
                    best = v - prev
                prev = v
        out.append(-1 if best == 101 else best)
    return out

for (nums, qs), want in examples:
    print(min_diff_prefix(nums, qs), "kỳ vọng", want)

[2, 1, 4, 1] kỳ vọng [2, 1, 4, 1]
[-1, 1, 1, 3] kỳ vọng [-1, 1, 1, 3]


**Độ phức tạp:** dựng `O(D · n)` với `D ≤ 100` số giá trị phân biệt; mỗi truy vấn `O(D)` ⇒ tổng `O(D · (n + q))` ≈ `1.2·10⁷`. Space `O(D · n)` — tới `10⁷` phần tử list Python (~80 MB): **đây là điểm yếu** của cách này trong Python.

Có thể giảm bộ nhớ bằng `array('i')` (4 byte/phần tử) — hoặc chuyển sang Cách 4.

### 7.3 Cách 4 — Danh sách vị trí + `bisect` (bộ nhớ `O(n)`)

Mỗi giá trị `v` lưu `pos[v]`. Kiểm tra có mặt bằng một `bisect_left`: `O(log n)` mỗi giá trị.

In [9]:
def min_diff_bisect(nums, queries):
    pos = [[] for _ in range(101)]
    for i, x in enumerate(nums):
        pos[x].append(i)
    vals = [v for v in range(1, 101) if pos[v]]
    out = []
    for l, r in queries:
        best, prev = 101, -1
        for v in vals:
            p = pos[v]
            k = bisect_left(p, l)
            if k < len(p) and p[k] <= r:
                if prev != -1 and v - prev < best:
                    best = v - prev
                prev = v
        out.append(-1 if best == 101 else best)
    return out

for (nums, qs), want in examples:
    print(min_diff_bisect(nums, qs), "kỳ vọng", want)

[2, 1, 4, 1] kỳ vọng [2, 1, 4, 1]
[-1, 1, 1, 3] kỳ vọng [-1, 1, 1, 3]


Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** dựng `O(n)`; mỗi truy vấn `O(D log n)` ⇒ tổng `O(n + q · D · log n)` ≈ `2·10⁴ · 100 · 17 = 3.4·10⁷` phép so sánh bên trong `bisect` (code C, nhanh). Space `O(n)`.

**Mẹo dừng sớm:** nếu `best == 1` thì không thể nhỏ hơn — thoát vòng `v`.

### 7.4 Cách 5 — Segment tree lưu bitmask

Mỗi nút lưu số nguyên có bit `v` bật nếu giá trị `v` xuất hiện trong đoạn của nút. Gộp = OR. Truy vấn `[l, r]` ⇒ mask `M` trong `O(log n)` phép OR trên số 101 bit; rồi tính hiệu nhỏ nhất giữa các bit bật kề nhau.

Ưu điểm: hỗ trợ **cập nhật điểm** `nums[i] = x` trong `O(log n)` (Bài 5).

In [10]:
def gap_of_mask(M):
    """Hiệu nhỏ nhất giữa hai bit bật kề nhau của M; -1 nếu < 2 bit."""
    best, prev = 101, -1
    while M:
        low = M & -M
        v = low.bit_length() - 1
        if prev != -1 and v - prev < best:
            best = v - prev
        prev = v
        M ^= low
    return -1 if best == 101 else best

def min_diff_segtree(nums, queries):
    n = len(nums)
    t = [0] * (2 * n)
    for i, x in enumerate(nums):
        t[n + i] = 1 << x
    for i in range(n - 1, 0, -1):
        t[i] = t[2 * i] | t[2 * i + 1]
    out = []
    for l, r in queries:
        M = 0; lo, hi = l + n, r + n + 1
        while lo < hi:
            if lo & 1: M |= t[lo]; lo += 1
            if hi & 1: hi -= 1; M |= t[hi]
            lo >>= 1; hi >>= 1
        out.append(gap_of_mask(M))
    return out

print(bin(0b10110100), "->", gap_of_mask(0b10110100))
for (nums, qs), want in examples:
    print(min_diff_segtree(nums, qs), "kỳ vọng", want)

0b10110100 -> 1
[2, 1, 4, 1] kỳ vọng [2, 1, 4, 1]
[-1, 1, 1, 3] kỳ vọng [-1, 1, 1, 3]


**Độ phức tạp:** dựng `O(n)`; truy vấn `O(log n + D)` ⇒ tổng `O(n + q (log n + D))`. Space `O(n)` số nguyên nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2, truy vấn `[3, 5]` và `[0, 2]`: in từng giá trị `v`, số lần trong đoạn, `prev`, `best`.

In [11]:
def trace_prefix(nums, l, r):
    vals = sorted(set(nums))
    pre = {v: [0] + list(accumulate(1 if x == v else 0 for x in nums)) for v in vals}
    best, prev = 101, -1
    print(f"đoạn nums[{l}..{r}] = {nums[l:r + 1]}")
    print(f"{'v':>3} | {'pre[v][r+1]':>11} | {'pre[v][l]':>9} | {'đếm':>3} | {'prev':>4} | best")
    for v in vals:
        c = pre[v][r + 1] - pre[v][l]
        note = ""
        if c > 0:
            if prev != -1 and v - prev < best:
                best = v - prev; note = f"  <- {v}-{prev}"
            prev = v
        print(f"{v:>3} | {pre[v][r + 1]:>11} | {pre[v][l]:>9} | {c:>3} | {prev:>4} | {best if best < 101 else '∞'}{note}")
    print("=> đáp án", -1 if best == 101 else best, "\n")

nums = [4, 5, 2, 2, 7, 10]
trace_prefix(nums, 3, 5)
trace_prefix(nums, 0, 2)

đoạn nums[3..5] = [2, 7, 10]
  v | pre[v][r+1] | pre[v][l] | đếm | prev | best
  2 |           2 |         1 |   1 |    2 | ∞
  4 |           1 |         1 |   0 |    2 | ∞
  5 |           1 |         1 |   0 |    2 | ∞
  7 |           1 |         0 |   1 |    7 | 5  <- 7-2
 10 |           1 |         0 |   1 |   10 | 3  <- 10-7
=> đáp án 3 

đoạn nums[0..2] = [4, 5, 2]
  v | pre[v][r+1] | pre[v][l] | đếm | prev | best
  2 |           1 |         0 |   1 |    2 | ∞
  4 |           1 |         0 |   1 |    4 | 2  <- 4-2
  5 |           1 |         0 |   1 |    5 | 1  <- 5-4
  7 |           0 |         0 |   0 |    5 | 1
 10 |           0 |         0 |   0 |    5 | 1
=> đáp án 1 



In [12]:
# Trace Cách 5: mask của truy vấn [3, 5] và các bit bật
n = len(nums); t = [0] * (2 * n)
for i, x in enumerate(nums): t[n + i] = 1 << x
for i in range(n - 1, 0, -1): t[i] = t[2 * i] | t[2 * i + 1]
lo, hi, M = 3 + n, 5 + n + 1, 0
while lo < hi:
    if lo & 1: print(f"  OR nút {lo} (bit {[v for v in range(101) if t[lo] >> v & 1]})"); M |= t[lo]; lo += 1
    if hi & 1: hi -= 1; print(f"  OR nút {hi} (bit {[v for v in range(101) if t[hi] >> v & 1]})"); M |= t[hi]
    lo >>= 1; hi >>= 1
print("bit bật của M:", [v for v in range(101) if M >> v & 1], "-> gap =", gap_of_mask(M))

  OR nút 9 (bit [2])
  OR nút 5 (bit [7, 10])
bit bật của M: [2, 7, 10] -> gap = 3


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def minDifference(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        pos = [[] for _ in range(101)]
        for i, x in enumerate(nums):
            pos[x].append(i)
        vals = [v for v in range(1, 101) if pos[v]]
        ans = []
        for l, r in queries:
            best, prev = 101, -1
            for v in vals:
                p = pos[v]
                k = bisect_left(p, l)
                if k < len(p) and p[k] <= r:
                    if prev != -1 and v - prev < best:
                        best = v - prev
                        if best == 1:
                            break                    # không thể nhỏ hơn 1
                    prev = v
            ans.append(-1 if best == 101 else best)
        return ans

sol = Solution()
for (nums, qs), want in examples:
    got = sol.minDifference(nums, qs)
    print(got, "OK" if got == want else "SAI")

[2, 1, 4, 1] OK
[-1, 1, 1, 3] OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Với tập `S` các giá trị phân biệt, `|S| ≥ 2`, sắp xếp `s₁ < s₂ < … < s_k`. Khi đó `min_{x≠y ∈ S} |x - y| = min_i (s_{i+1} - s_i)`.

*Chứng minh:* với `i < j`, `s_j - s_i = Σ_{t=i}^{j-1} (s_{t+1} - s_t) ≥ s_{i+1} - s_i` (mọi số hạng dương). Nên cặp tối ưu luôn là cặp kề. ✓

**Bổ đề 2.** Hiệu tuyệt đối nhỏ nhất của `nums[l..r]` (theo định nghĩa đề, chỉ xét cặp khác giá trị) bằng Bổ đề 1 áp cho `S(l, r)` — tập giá trị có mặt; bằng `-1` khi `|S(l, r)| = 1`. Vì mọi cặp `i < j` với `nums[i] ≠ nums[j]` cho ra một cặp `x ≠ y ∈ S`, và ngược lại mỗi cặp `x ≠ y ∈ S` đến từ ít nhất một cặp chỉ số.

**Cách 3/4/5** đều tính đúng `S(l, r)`:
- Cách 3: `v ∈ S ⇔ pre[v][r+1] - pre[v][l] > 0` (định nghĩa prefix).
- Cách 4: `v ∈ S ⇔ ∃ p ∈ pos[v], l ≤ p ≤ r ⇔` phần tử đầu tiên `≥ l` tồn tại và `≤ r` (`pos[v]` tăng dần).
- Cách 5: mask của đoạn = OR các mask của các nút phủ rời rạc đoạn — đúng bằng hợp các tập.

Sau đó duyệt `v` **tăng dần**, giữ `prev` = phần tử trước trong `S` ⇒ các hiệu `v - prev` chính là các hiệu kề của Bổ đề 1. ✓

**Dừng sớm ở `best = 1`:** hiệu giữa hai số nguyên khác nhau ≥ 1 ⇒ không thể cải thiện.

## 11. Phân tích Time / Space

`n ≤ 10⁵`, `q ≤ 2·10⁴`, `D ≤ 100` số giá trị phân biệt.

| Cách | Dựng | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Mọi cặp | — | `O(len²)` | `O(1)` |
| 2. Sắp xếp đoạn | — | `O(len log len)` | `O(len)` |
| 3. Prefix count | `O(D n)` | `O(D)` | `O(D n)` |
| 4. Vị trí + bisect | `O(n)` | `O(D log n)` | `O(n)` |
| 5. Segment tree mask | `O(n)` | `O(log n + D)` | `O(n)` |

Đo thực tế với `n = 10⁵`, `q = 2·10⁴` truy vấn ngẫu nhiên (Cách 3 dùng bản ít giá trị để giữ bộ nhớ vừa phải):

In [14]:
rng = random.Random(1)
n, q = 10**5, 2 * 10**4
big_nums = [rng.randint(1, 100) for _ in range(n)]
big_q = []
for _ in range(q):
    l = rng.randrange(n - 1); r = rng.randrange(l + 1, n); big_q.append([l, r])

# Trường hợp khó cho dừng sớm: chỉ các giá trị chẵn ⇒ best ≥ 2, không bao giờ dừng ở 1
even_nums = [2 * rng.randint(1, 50) for _ in range(n)]

res = {}
for name, fn in [("bisect", min_diff_bisect), ("segtree", min_diff_segtree), ("submit", Solution().minDifference)]:
    t0 = time.perf_counter(); a = fn(big_nums, big_q); b = fn(even_nums, big_q)
    res[name] = (a, b); print(f"{name:8s}: {time.perf_counter() - t0:.2f}s")
assert res["bisect"] == res["segtree"] == res["submit"]

small_vals = [rng.choice([3, 10, 17, 40, 41]) for _ in range(n)]           # D = 5 cho Cách 3
t0 = time.perf_counter(); a = min_diff_prefix(small_vals, big_q); print(f"prefix (D=5): {time.perf_counter() - t0:.2f}s")
assert a == Solution().minDifference(small_vals, big_q)

t0 = time.perf_counter(); min_diff_sort(big_nums, big_q[:300]); print(f"sort, chỉ 300 truy vấn: {time.perf_counter() - t0:.2f}s")

bisect  : 0.78s


segtree : 0.38s


submit  : 0.29s
prefix (D=5): 0.05s
sort, chỉ 300 truy vấn: 0.04s


## 12. Bộ test

In [15]:
fns = [min_diff_sort, min_diff_prefix, min_diff_bisect, min_diff_segtree, lambda a, b: Solution().minDifference(a, b)]

for (nums, qs), want in examples:
    assert min_diff_brute(nums, qs) == want
    for fn in fns:
        assert fn(nums, qs) == want

edge = [
    (([1, 1], [[0, 1]]), [-1]),
    (([1, 100], [[0, 1]]), [99]),
    (([100, 1, 100], [[0, 2], [0, 1], [1, 2]]), [99, 99, 99]),
    (([5, 5, 5, 6], [[0, 2], [0, 3], [2, 3]]), [-1, 1, 1]),
    (([1, 3], [[0, 1]]), [2]),                       # bẫy prev khởi tạo bằng 0
    (([2, 9, 2, 9, 5], [[0, 3], [1, 4], [0, 4]]), [7, 3, 3]),
]
for (nums, qs), want in edge:
    assert min_diff_brute(nums, qs) == want, (nums, qs)
    for fn in fns:
        assert fn(nums, qs) == want, (fn, nums, qs)

rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(2, 25); hi = rng.choice([2, 5, 20, 100])
    nums = [rng.randint(1, hi) for _ in range(n)]
    qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randrange(n - 1); qs.append([l, rng.randrange(l + 1, n)])
    want = min_diff_brute(nums, qs)
    for fn in fns:
        assert fn(nums, qs) == want, (fn, nums, qs)
print("ví dụ + edge + 500 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 500 bộ ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tính cả hiệu `0` giữa hai phần tử bằng nhau

In [16]:
def wrong_with_zero(nums, queries):
    out = []
    for l, r in queries:
        a = sorted(nums[l:r + 1])                         # SAI: không bỏ trùng
        out.append(min(y - x for x, y in zip(a, a[1:])))
    return out

for (nums, qs), want in examples:
    print("sai =", wrong_with_zero(nums, qs), "| đúng =", want)

sai = [2, 1, 4, 1] | đúng = [2, 1, 4, 1]
sai = [0, 1, 0, 3] | đúng = [-1, 1, 1, 3]


Đề yêu cầu `a[i] ≠ a[j]`. **Sửa:** dùng `set` (hoặc tập giá trị có mặt), và trả `-1` khi chỉ còn một giá trị.

### Lỗi 2 — Off-by-one trong prefix: `pre[r] - pre[l]`

In [17]:
def wrong_prefix_offbyone(nums, queries):
    vals = sorted(set(nums))
    pre = {v: [0] + list(accumulate(1 if x == v else 0 for x in nums)) for v in vals}
    out = []
    for l, r in queries:
        best, prev = 101, -1
        for v in vals:
            if pre[v][r] - pre[v][l] > 0:                 # SAI: bỏ sót nums[r]
                if prev != -1: best = min(best, v - prev)
                prev = v
        out.append(-1 if best == 101 else best)
    return out

for (nums, qs), want in examples:
    print("sai =", wrong_prefix_offbyone(nums, qs), "| đúng =", want)

sai = [-1, -1, -1, 1] | đúng = [2, 1, 4, 1]
sai = [-1, 1, 1, 5] | đúng = [-1, 1, 1, 3]


Với `pre` có phần tử `0` đầu, `pre[i]` đếm các chỉ số `< i` ⇒ đoạn `[l, r]` là `pre[r + 1] - pre[l]`. **Sửa:** `r + 1`.

### Lỗi 3 — Khởi tạo `prev = 0`

In [18]:
def wrong_prev_zero(nums, queries):
    pos = [[] for _ in range(101)]
    for i, x in enumerate(nums): pos[x].append(i)
    out = []
    for l, r in queries:
        best, prev = 101, 0                               # SAI: coi như giá trị 0 "có mặt"
        for v in range(1, 101):
            p = pos[v]; k = bisect_left(p, l)
            if k < len(p) and p[k] <= r:
                best = min(best, v - prev); prev = v
        out.append(-1 if best == 101 else best)
    return out

for (nums, qs), want in examples + [(([1, 3], [[0, 1]]), [2]), (([7, 7], [[0, 1]]), [-1])]:
    print("sai =", wrong_prev_zero(nums, qs), "| đúng =", want)

sai = [1, 1, 4, 1] | đúng = [2, 1, 4, 1]
sai = [2, 1, 1, 2] | đúng = [-1, 1, 1, 3]
sai = [1] | đúng = [2]
sai = [7] | đúng = [-1]


`v - 0 = v` bị tính như một hiệu thật: `[1, 3]` cho `1` thay vì `2`; `[7, 7]` cho `7` thay vì `-1`. **Sửa:** `prev = -1` (lính canh) và chỉ cập nhật khi `prev != -1`.

### Lỗi 4 — Chỉ so các phần tử **kề nhau trong mảng** (không sắp xếp)

In [19]:
def wrong_adjacent_in_array(nums, queries):
    out = []
    for l, r in queries:
        ds = [abs(nums[i + 1] - nums[i]) for i in range(l, r) if nums[i] != nums[i + 1]]
        out.append(min(ds) if ds else -1)
    return out

rng = random.Random(5)
for _ in range(10**4):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(3, 5))]
    qs = [[0, len(nums) - 1]]
    if wrong_adjacent_in_array(nums, qs) != min_diff_brute(nums, qs):
        print("phản ví dụ:", nums, "sai =", wrong_adjacent_in_array(nums, qs), "| đúng =", min_diff_brute(nums, qs))
        break
print("ví dụ 1 đề:", wrong_adjacent_in_array(*examples[0][0]), "| đúng =", examples[0][1])

phản ví dụ: [8, 4, 7, 9] sai = [2] | đúng = [1]
ví dụ 1 đề: [2, 1, 4, 1] | đúng = [2, 1, 4, 1]


Hai giá trị gần nhau có thể nằm xa nhau trong mảng. **Sửa:** làm việc trên **tập giá trị đã sắp xếp**.

### Lỗi 5 — Dựng `pre` 2 chiều kích thước `n × 101` bằng list Python

Chạy đúng, nhưng `10⁵ × 101` ô list ≈ `10⁷` con trỏ ⇒ ~80 MB và dựng chậm (vòng lặp `n · 101` trong Python ≈ vài giây). **Sửa:** chỉ dựng cho các giá trị xuất hiện, hoặc dùng Cách 4/5 với bộ nhớ `O(n)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Miền giá trị nhỏ** (≤ 100, ≤ 26 chữ cái…) + truy vấn đoạn ⇒ prefix count **theo giá trị**, mỗi truy vấn duyệt cả miền.
- "Hiệu nhỏ nhất của một tập" ⇒ sắp xếp, so phần tử kề; trên miền nhỏ thì "sắp xếp" = duyệt `v` tăng dần.
- Bitmask số nguyên lớn của Python là "bitset" miễn phí: OR/AND nhanh, `M & -M` lấy bit thấp nhất.
- Luôn nghĩ tới trường hợp **chỉ một giá trị** ⇒ `-1`.
- Họ bài: 530/783 *Minimum Distance Between BST Nodes*, 1200 *Minimum Absolute Difference*, 1177 *Can Make Palindrome from Substring* (prefix theo chữ cái), 2055 *Plates Between Candles*.

## 15. Biến thể của bài

1. **Giá trị lớn (tới `10⁹`)**: miền không còn nhỏ — cần kỹ thuật khác (Mo's algorithm + cấu trúc có thứ tự, hoặc offline phức tạp hơn).
2. **Đếm số giá trị phân biệt** trong đoạn với miền nhỏ: cùng prefix count, đếm `v` có mặt (Bài 4).
3. **Hiệu lớn nhất** giữa hai giá trị phân biệt: `max(S) - min(S)` — chỉ cần min/max đoạn (sparse table).
4. **Có cập nhật điểm** `nums[i] = x`: segment tree bitmask của Cách 5 (Bài 5).

In [20]:
def max_distinct_diff(nums, queries):
    """Biến thể 3 (miền nhỏ): max(S) - min(S), -1 nếu |S| < 2."""
    pos = [[] for _ in range(101)]
    for i, x in enumerate(nums): pos[x].append(i)
    out = []
    for l, r in queries:
        have = [v for v in range(1, 101) if pos[v] and (k := bisect_left(pos[v], l)) < len(pos[v]) and pos[v][k] <= r]
        out.append(have[-1] - have[0] if len(have) > 1 else -1)
    return out

print(max_distinct_diff([4, 5, 2, 2, 7, 10], [[2, 3], [0, 2], [0, 5], [3, 5]]))

[-1, 3, 8, 8]


## 16. So sánh các cách

| Cách | Ý tưởng | Tổng thời gian (tệ nhất) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | định nghĩa | `O(q n²)` | `O(1)` | kiểm chứng |
| 2. Sắp xếp đoạn | Bổ đề 1 | `O(q n log n)` | `O(n)` | đoạn ngắn |
| 3. Prefix count | có mặt `O(1)` | `O(D (n + q))` | `O(D n)` | lời giải "chuẩn" ngôn ngữ biên dịch |
| 4. Vị trí + bisect | có mặt `O(log n)` | `O(n + q D log n)` | `O(n)` | **nộp bài Python** |
| 5. Segtree bitmask | OR tập | `O(n + q (log n + D))` | `O(n)` | có cập nhật |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [7, 7, 7, 1, 50, 48]`, `queries = [[0, 2], [0, 3], [3, 5], [2, 4]]`? (Ví dụ tự đặt.)

**Bài 2 (tính tay):** với ví dụ 1 (`nums = [1,3,4,8]`), viết `pre[v]` cho `v ∈ {1, 3, 4, 8}` và dùng nó trả lời truy vấn `[1, 3]`.

**Bài 3 (điền TODO):** hoàn thiện `gap_of_set(present)` — nhận list bool độ dài 101 (`present[v]` = giá trị `v` có mặt) và trả về đáp án theo định nghĩa đề.

**Bài 4 (tự viết lại):** viết `count_distinct(nums, queries)` trả về số giá trị phân biệt trong mỗi đoạn, dùng prefix count theo giá trị (`nums[i] ≤ 100`).

**Bài 5 (mini challenge):** cho thêm thao tác cập nhật: danh sách `ops`, mỗi phần tử là `("set", i, x)` hoặc `("ask", l, r)`. Trả về đáp án của các `"ask"` theo thứ tự. Dùng segment tree bitmask (`O(log n)` mỗi thao tác + `O(D)` tính gap). (Biến thể tự đặt.)

In [21]:
def gap_of_set(present):
    # TODO: duyệt v = 1..100, giữ prev = -1, best = 101; trả -1 nếu best không đổi
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Tìm tập giá trị có mặt của từng đoạn. (2) `[0,2] → {7}`, `[0,3] → {1, 7}`, `[3,5] → {1, 48, 50}`, `[2,4] → {1, 7, 50}`. (3) Áp hiệu kề.

**Bài 2:** (1) `pre[1] = [0,1,1,1,1]`, `pre[3] = [0,0,1,1,1]`. (2) `pre[4] = [0,0,0,1,1]`, `pre[8] = [0,0,0,0,1]`. (3) Truy vấn `[1,3]`: `pre[v][4] - pre[v][1]` cho `1→0, 3→1, 4→1, 8→1` ⇒ `{3,4,8}` ⇒ `1`.

**Bài 3:** (1) Giống vòng trong của Cách 3. (2) Chỉ cập nhật `best` khi `prev != -1`. (3) Có thể `break` khi `best == 1`.

**Bài 4:** (1) Như Cách 3 nhưng đếm số `v` có `pre[v][r+1] - pre[v][l] > 0`. (2) Kiểm chứng bằng `len(set(nums[l:r+1]))`. (3) Chỉ dựng `pre` cho giá trị xuất hiện.

**Bài 5:** (1) Lá `n + i` lưu `1 << nums[i]`. (2) Cập nhật: gán lá, đi lên `i //= 2` và tính lại `t[i] = t[2i] | t[2i+1]`. (3) Truy vấn như Cách 5 rồi `gap_of_mask`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
b1 = ([7, 7, 7, 1, 50, 48], [[0, 2], [0, 3], [3, 5], [2, 4]])
assert Solution().minDifference(*b1) == min_diff_brute(*b1) == [-1, 6, 2, 6]

# Bài 2
nums = [1, 3, 4, 8]
pre = {v: [0] + list(accumulate(1 if x == v else 0 for x in nums)) for v in (1, 3, 4, 8)}
print("pre:", pre)
assert [v for v in (1, 3, 4, 8) if pre[v][4] - pre[v][1] > 0] == [3, 4, 8]
assert Solution().minDifference(nums, [[1, 3]]) == [1]

# Bài 3
def gap_of_set_ans(present):
    best, prev = 101, -1
    for v in range(1, 101):
        if present[v]:
            if prev != -1 and v - prev < best:
                best = v - prev
            prev = v
    return -1 if best == 101 else best

rng = random.Random(31)
for _ in range(500):
    arr = [rng.randint(1, rng.choice([3, 100])) for _ in range(rng.randint(1, 8))]
    present = [False] * 101
    for x in arr: present[x] = True
    assert gap_of_set_ans(present) == min_abs_diff(arr)

# Bài 4
def count_distinct_ans(nums, queries):
    vals = sorted(set(nums))
    pre = {v: [0] + list(accumulate(1 if x == v else 0 for x in nums)) for v in vals}
    return [sum(1 for v in vals if pre[v][r + 1] - pre[v][l] > 0) for l, r in queries]

for _ in range(300):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(2, 20))]
    qs = []
    for _ in range(5):
        l = rng.randrange(len(nums) - 1); qs.append([l, rng.randrange(l + 1, len(nums))])
    assert count_distinct_ans(nums, qs) == [len(set(nums[l:r + 1])) for l, r in qs]
print("Bài 4 ví dụ:", count_distinct_ans([4, 5, 2, 2, 7, 10], [[2, 3], [0, 2], [0, 5], [3, 5]]))
print("Bài 1-4: OK")

pre: {1: [0, 1, 1, 1, 1], 3: [0, 0, 1, 1, 1], 4: [0, 0, 0, 1, 1], 8: [0, 0, 0, 0, 1]}
Bài 4 ví dụ: [1, 3, 5, 3]
Bài 1-4: OK


In [23]:
# Bài 5 — segment tree bitmask có cập nhật điểm
def process_ops(nums, ops):
    n = len(nums); t = [0] * (2 * n)
    for i, x in enumerate(nums): t[n + i] = 1 << x
    for i in range(n - 1, 0, -1): t[i] = t[2 * i] | t[2 * i + 1]
    out = []
    for op in ops:
        if op[0] == "set":
            _, i, x = op; i += n; t[i] = 1 << x; i >>= 1
            while i:
                t[i] = t[2 * i] | t[2 * i + 1]; i >>= 1
        else:
            _, l, r = op; M = 0; lo, hi = l + n, r + n + 1
            while lo < hi:
                if lo & 1: M |= t[lo]; lo += 1
                if hi & 1: hi -= 1; M |= t[hi]
                lo >>= 1; hi >>= 1
            out.append(gap_of_mask(M))
    return out

def process_ops_brute(nums, ops):
    a = nums[:]; out = []
    for op in ops:
        if op[0] == "set": a[op[1]] = op[2]
        else: out.append(min_abs_diff(a[op[1]:op[2] + 1]))
    return out

rng = random.Random(55)
for _ in range(300):
    n = rng.randint(2, 15); nums = [rng.randint(1, 12) for _ in range(n)]; ops = []
    for _ in range(rng.randint(1, 15)):
        if rng.random() < 0.4: ops.append(("set", rng.randrange(n), rng.randint(1, 12)))
        else:
            l = rng.randrange(n - 1); ops.append(("ask", l, rng.randrange(l + 1, n)))
    assert process_ops(nums, ops) == process_ops_brute(nums, ops)

demo = [("ask", 0, 5), ("set", 1, 2), ("ask", 0, 2), ("set", 0, 2), ("ask", 0, 3)]
print("Bài 5 ví dụ:", process_ops([4, 5, 2, 2, 7, 10], demo), "| brute:", process_ops_brute([4, 5, 2, 2, 7, 10], demo))

Bài 5 ví dụ: [1, 2, -1] | brute: [1, 2, -1]


## 20. Template nhớ nhanh + Checklist

```python
# Truy vấn "hiệu nhỏ nhất giữa các giá trị phân biệt" trên đoạn, miền giá trị nhỏ (≤ 100)
pos = [[] for _ in range(101)]
for i, x in enumerate(nums): pos[x].append(i)
vals = [v for v in range(1, 101) if pos[v]]
for l, r in queries:
    best, prev = 101, -1                         # prev = -1 là lính canh
    for v in vals:                               # tăng dần = "sắp xếp đếm"
        k = bisect_left(pos[v], l)
        if k < len(pos[v]) and pos[v][k] <= r:   # v có mặt trong [l, r]
            if prev != -1: best = min(best, v - prev)
            prev = v
    ans.append(-1 if best == 101 else best)
# Prefix count tương đương: có mặt ⇔ pre[v][r + 1] - pre[v][l] > 0
```

**Checklist trước khi submit:**
- [ ] Bỏ qua cặp bằng nhau (không có hiệu `0`).
- [ ] Trả `-1` khi đoạn chỉ có một giá trị.
- [ ] Prefix dùng `pre[r + 1] - pre[l]`.
- [ ] `prev` khởi tạo là lính canh (`-1`), không phải `0`.
- [ ] Duyệt `v` tăng dần; có thể dừng sớm khi `best == 1`.
- [ ] Bộ nhớ: tránh bảng `n × 101` bằng list Python.